# Problem 3 - Data Parallelism in PyTorch

This notebook completes Problem 3 on CIFAR-10 using the `kuangliu/pytorch-cifar` ResNet-18 architecture. It is designed for an HPC OOD session with up to 4 visible GPUs.

The notebook is organized around the homework parts:
- Part 1: single-GPU batch-size sweep and compute-only timing
- Part 2: 2-GPU and 4-GPU `DataParallel` timing with speedup analysis
- Part 3: compute-versus-communication breakdown
- Part 4: all-reduce communication model and bandwidth-utilization analysis


## Experiment Setup

The CIFAR-10 preprocessing, optimizer, and model match the prompt:
- train transform: random crop `32x32` with padding `4`, random horizontal flip, tensor conversion, normalization
- test transform: tensor conversion and normalization
- optimizer: SGD with learning rate `0.1`, momentum `0.9`, and weight decay `5e-4`
- training DataLoader `num_workers=2`, testing DataLoader batch size `100`

For every timing run, the notebook executes 2 epochs and reports epoch 2 only. Part 1 excludes DataLoader wait time. Part 2 includes the full epoch wall time, as required by the prompt.


In [ ]:
from __future__ import annotations

import gc
import os
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', str(Path.cwd() / '.mpl-cache'))

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.backends.cudnn as cudnn
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from IPython.display import Markdown, display
from torch.utils.data import DataLoader

SEED = 301
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

cudnn.benchmark = True

DEFAULT_DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
DATA_ROOT = Path(os.environ.get('CIFAR10_ROOT', './data/cifar10'))
DOWNLOAD_CIFAR10 = os.environ.get('CIFAR10_DOWNLOAD', '1') != '0'
NUM_WORKERS = 2
TRAIN_BATCH_START = 32
BATCH_MULTIPLIER = 4
MAX_BATCH_SIZE = 32768
MANUAL_PEAK_BW_GBPS = None

print(f'torch={torch.__version__}')
print(f'torchvision={torchvision.__version__}')
print(f'cuda_available={torch.cuda.is_available()}')
print(f'visible_gpu_count={torch.cuda.device_count()}')
print(f'default_device={DEFAULT_DEVICE}')
print(f'data_root={DATA_ROOT.resolve()}')
print(f'download_cifar10={DOWNLOAD_CIFAR10}')
if torch.cuda.is_available():
    for idx in range(torch.cuda.device_count()):
        print(f'gpu_{idx}={torch.cuda.get_device_name(idx)}')
else:
    print('A CUDA device is required for the homework timing runs.')


def format_table_for_markdown(df: pd.DataFrame, floatfmt: str = '.3f') -> str:
    try:
        return df.to_markdown(index=False, floatfmt=floatfmt)
    except Exception:
        return '```\n' + df.to_string(index=False) + '\n```'


In [ ]:
@dataclass
class TimingResult:
    part: str
    gpu_count: int
    per_gpu_batch_size: int
    global_batch_size: int
    status: str
    warmup_wall_time_sec: float | None
    warmup_step_time_sec: float | None
    epoch2_wall_time_sec: float | None
    epoch2_step_time_sec: float | None
    images_per_sec_wall: float | None
    avg_loss_epoch2: float | None
    avg_acc_epoch2: float | None
    num_examples_epoch2: int | None
    num_steps_epoch2: int | None
    oom_message: str | None = None


def generate_batch_sizes(start: int = TRAIN_BATCH_START, multiplier: int = BATCH_MULTIPLIER, max_batch_size: int = MAX_BATCH_SIZE) -> list[int]:
    values: list[int] = []
    current = start
    while current <= max_batch_size:
        values.append(current)
        current *= multiplier
    return values


def cleanup_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def assert_cuda(device: str | torch.device) -> torch.device:
    resolved = torch.device(device)
    if resolved.type != 'cuda' or not torch.cuda.is_available():
        raise RuntimeError("Problem 3 must be run on a GPU node with CUDA available.")
    return resolved


def ensure_gpu_count(required: int) -> None:
    available = torch.cuda.device_count()
    if available < required:
        raise RuntimeError(f'Requested {required} GPUs but only {available} visible GPUs are available in this session.')


def get_device_ids(num_gpus: int) -> list[int]:
    ensure_gpu_count(num_gpus)
    return list(range(num_gpus))


def count_trainable_parameters(model: nn.Module) -> int:
    return sum(param.numel() for param in model.parameters() if param.requires_grad)


## Data Pipeline

The training set uses the exact transformation order requested in the prompt. The testing DataLoader is created as well so the notebook has the full Problem 3 input pipeline, even though the main measurements are training-only.


In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2023, 0.1994, 0.2010)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])


def make_dataloaders(
    train_batch_size: int,
    test_batch_size: int = 100,
    data_root: str | Path = DATA_ROOT,
    num_workers: int = NUM_WORKERS,
    download: bool = DOWNLOAD_CIFAR10,
) -> tuple[DataLoader, DataLoader]:
    data_root = Path(data_root)
    train_set = torchvision.datasets.CIFAR10(root=data_root, train=True, download=download, transform=train_transform)
    test_set = torchvision.datasets.CIFAR10(root=data_root, train=False, download=download, transform=test_transform)

    loader_kwargs = {
        'num_workers': num_workers,
        'pin_memory': torch.cuda.is_available(),
        'persistent_workers': num_workers > 0,
    }

    train_loader = DataLoader(train_set, batch_size=train_batch_size, shuffle=True, **loader_kwargs)
    test_loader = DataLoader(test_set, batch_size=test_batch_size, shuffle=False, **loader_kwargs)
    return train_loader, test_loader


## ResNet-18 Model

This is the CIFAR-style ResNet used by `kuangliu/pytorch-cifar`: a `3x3` input stem, residual blocks, and a final average-pooling plus linear classification head.


In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes: int, planes: int, stride: int = 1) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_planes: int, planes: int, stride: int = 1) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, self.expansion * planes, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(self.expansion * planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNet(nn.Module):
    def __init__(self, block: type[nn.Module], num_blocks: list[int], num_classes: int = 10) -> None:
        super().__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.linear = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block: type[nn.Module], planes: int, num_blocks: int, stride: int) -> nn.Sequential:
        strides = [stride] + [1] * (num_blocks - 1)
        layers: list[nn.Module] = []
        for stride_value in strides:
            layers.append(block(self.in_planes, planes, stride_value))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)
        out = torch.flatten(out, 1)
        out = self.linear(out)
        return out


def ResNet18() -> ResNet:
    return ResNet(BasicBlock, [2, 2, 2, 2])


def build_model(device: str | torch.device = DEFAULT_DEVICE, num_gpus: int = 1) -> nn.Module:
    resolved = assert_cuda(device)
    model = ResNet18().to(resolved)
    if num_gpus > 1:
        model = nn.DataParallel(model, device_ids=get_device_ids(num_gpus))
    return model


model = ResNet18()
print(model.__class__.__name__)
print(f'trainable_parameters={count_trainable_parameters(model):,}')


## Timing Helpers

The training helper records two timings per epoch:
- `wall_time_sec`: full epoch time, including DataLoader wait time
- `step_time_sec`: time spent after a batch is produced by the DataLoader, including CPU-to-GPU transfer, forward pass, backpropagation, optimizer update, and any synchronization or communication inside `DataParallel`


In [ ]:
def build_optimizer(model: nn.Module) -> torch.optim.Optimizer:
    return torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)


def train_one_epoch_with_timing(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, criterion: nn.Module, device: str | torch.device) -> dict[str, float | int]:
    resolved = assert_cuda(device)
    model.train()

    total_examples = 0
    total_loss = 0.0
    total_correct = 0
    step_time_sec = 0.0
    num_steps = 0

    torch.cuda.synchronize(device=resolved)
    epoch_start = time.perf_counter()
    for inputs, targets in loader:
        torch.cuda.synchronize(device=resolved)
        step_start = time.perf_counter()

        inputs = inputs.to(resolved, non_blocking=True)
        targets = targets.to(resolved, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        torch.cuda.synchronize(device=resolved)
        step_end = time.perf_counter()

        batch_size = targets.size(0)
        num_steps += 1
        step_time_sec += step_end - step_start
        total_examples += batch_size
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == targets).sum().item()

    torch.cuda.synchronize(device=resolved)
    epoch_end = time.perf_counter()
    return {
        'wall_time_sec': epoch_end - epoch_start,
        'step_time_sec': step_time_sec,
        'avg_loss': total_loss / total_examples,
        'avg_acc': total_correct / total_examples,
        'num_examples': total_examples,
        'num_steps': num_steps,
    }


def run_training_configuration(part: str, per_gpu_batch_size: int, gpu_count: int, device: str | torch.device = DEFAULT_DEVICE, data_root: str | Path = DATA_ROOT, download: bool = DOWNLOAD_CIFAR10) -> TimingResult:
    resolved = assert_cuda(device)
    ensure_gpu_count(gpu_count)
    global_batch_size = per_gpu_batch_size * gpu_count
    cleanup_cuda()
    try:
        train_loader, _ = make_dataloaders(train_batch_size=global_batch_size, data_root=data_root, num_workers=NUM_WORKERS, download=download)
        model = build_model(device=resolved, num_gpus=gpu_count)
        criterion = nn.CrossEntropyLoss()
        optimizer = build_optimizer(model)

        warmup_metrics = train_one_epoch_with_timing(model, train_loader, optimizer, criterion, resolved)
        measured_metrics = train_one_epoch_with_timing(model, train_loader, optimizer, criterion, resolved)

        return TimingResult(
            part=part,
            gpu_count=gpu_count,
            per_gpu_batch_size=per_gpu_batch_size,
            global_batch_size=global_batch_size,
            status='ok',
            warmup_wall_time_sec=float(warmup_metrics['wall_time_sec']),
            warmup_step_time_sec=float(warmup_metrics['step_time_sec']),
            epoch2_wall_time_sec=float(measured_metrics['wall_time_sec']),
            epoch2_step_time_sec=float(measured_metrics['step_time_sec']),
            images_per_sec_wall=float(measured_metrics['num_examples']) / float(measured_metrics['wall_time_sec']),
            avg_loss_epoch2=float(measured_metrics['avg_loss']),
            avg_acc_epoch2=float(measured_metrics['avg_acc']),
            num_examples_epoch2=int(measured_metrics['num_examples']),
            num_steps_epoch2=int(measured_metrics['num_steps']),
        )
    except RuntimeError as exc:
        if 'out of memory' not in str(exc).lower():
            raise
        cleanup_cuda()
        return TimingResult(
            part=part,
            gpu_count=gpu_count,
            per_gpu_batch_size=per_gpu_batch_size,
            global_batch_size=global_batch_size,
            status='oom',
            warmup_wall_time_sec=None,
            warmup_step_time_sec=None,
            epoch2_wall_time_sec=None,
            epoch2_step_time_sec=None,
            images_per_sec_wall=None,
            avg_loss_epoch2=None,
            avg_acc_epoch2=None,
            num_examples_epoch2=None,
            num_steps_epoch2=None,
            oom_message=str(exc),
        )
    finally:
        cleanup_cuda()


def run_part1_single_gpu_sweep(batch_sizes: list[int] | None = None, device: str | torch.device = DEFAULT_DEVICE) -> pd.DataFrame:
    batch_sizes = batch_sizes or generate_batch_sizes()
    rows: list[dict] = []
    for batch_size in batch_sizes:
        print(f'Running Part 1 batch_size={batch_size} on 1 GPU ...')
        result = run_training_configuration(part='part1', per_gpu_batch_size=batch_size, gpu_count=1, device=device)
        rows.append(asdict(result))
        if result.status == 'ok':
            print(f"epoch2_step_time={result.epoch2_step_time_sec:.3f}s, epoch2_wall_time={result.epoch2_wall_time_sec:.3f}s, images_per_sec={result.images_per_sec_wall:.2f}")
        else:
            print(f'OOM at batch_size={batch_size}. Stopping the single-GPU sweep.')
            break
    return pd.DataFrame(rows)


def run_single_gpu_wall_baseline(batch_sizes: list[int], device: str | torch.device = DEFAULT_DEVICE) -> pd.DataFrame:
    rows: list[dict] = []
    for batch_size in batch_sizes:
        print(f'Running Part 2 single-GPU wall baseline for batch_size={batch_size} ...')
        result = run_training_configuration(part='part2_baseline', per_gpu_batch_size=batch_size, gpu_count=1, device=device)
        rows.append(asdict(result))
    return pd.DataFrame(rows)


def run_dataparallel_sweep(per_gpu_batch_sizes: list[int], gpu_counts: tuple[int, ...] = (2, 4), device: str | torch.device = DEFAULT_DEVICE) -> pd.DataFrame:
    rows: list[dict] = []
    for gpu_count in gpu_counts:
        ensure_gpu_count(gpu_count)
        for batch_size in per_gpu_batch_sizes:
            print(f'Running DataParallel with {gpu_count} GPUs and per_gpu_batch_size={batch_size} ...')
            result = run_training_configuration(part='part2_3', per_gpu_batch_size=batch_size, gpu_count=gpu_count, device=device)
            rows.append(asdict(result))
            if result.status == 'ok':
                print(f"epoch2_wall_time={result.epoch2_wall_time_sec:.3f}s, epoch2_step_time={result.epoch2_step_time_sec:.3f}s, images_per_sec={result.images_per_sec_wall:.2f}")
            else:
                print(f'OOM for gpu_count={gpu_count}, per_gpu_batch_size={batch_size}.')
    return pd.DataFrame(rows)


## Part 1 - Single-GPU Batch-Size Sweep

This section finds the largest per-GPU batch size that fits on one GPU and records both compute-only time and full-epoch wall time for reuse in later parts.


In [ ]:
batch_sizes = generate_batch_sizes()
print('Planned Part 1 batch sizes:', batch_sizes)
part1_results = run_part1_single_gpu_sweep(batch_sizes=batch_sizes, device=DEFAULT_DEVICE)
part1_results


In [ ]:
part1_successful = part1_results[part1_results['status'] == 'ok'].copy().sort_values('per_gpu_batch_size')
if part1_successful.empty:
    print('No successful Part 1 runs yet. Execute the sweep cell above on a CUDA node first.')
else:
    display(part1_successful)
    ax = part1_successful.plot(
        x='per_gpu_batch_size',
        y='epoch2_step_time_sec',
        marker='o',
        legend=False,
        figsize=(7, 4),
        title='Problem 3 Part 1: Single-GPU Epoch-2 Compute Time',
    )
    ax.set_xlabel('Per-GPU batch size')
    ax.set_ylabel('Epoch 2 compute time (sec)')
    ax.set_xscale('log', base=4)
    ax.grid(True, alpha=0.3)
    plt.show()


In [ ]:
def build_part1_response(results: pd.DataFrame) -> str:
    successful = results[results['status'] == 'ok'].copy().sort_values('per_gpu_batch_size')
    if successful.empty:
        return 'Run Part 1 first so the notebook can summarize the single-GPU measurements.'

    batch_list = ', '.join(str(int(value)) for value in successful['per_gpu_batch_size'])
    largest_fit = int(successful['per_gpu_batch_size'].max())
    fastest_row = successful.loc[successful['epoch2_step_time_sec'].idxmin()]
    oom_rows = results[results['status'] == 'oom']

    lines = [
        '## Part 1 Response',
        '',
        'I trained ResNet-18 on CIFAR-10 using a single GPU (`cuda:0`) and ran 2 epochs for each batch size. Epoch 1 was treated as a warmup pass, and I reported epoch 2 only.',
        'The Part 1 timing starts after each batch is fetched from the DataLoader, so it excludes DataLoader wait time but includes CPU-to-GPU transfer, forward propagation, backpropagation, and the SGD weight update.',
        f'The successful per-GPU batch sizes were {batch_list}. The largest batch size that fit on one GPU was {largest_fit}.',
        f'The fastest successful configuration in compute-only epoch-2 time was batch size {int(fastest_row["per_gpu_batch_size"])} with {fastest_row["epoch2_step_time_sec"]:.3f} seconds.',
    ]
    if not oom_rows.empty:
        failed_batch = int(oom_rows.iloc[0]['per_gpu_batch_size'])
        lines.append(f'The next 4x increase failed at batch size {failed_batch} with an out-of-memory error, so the sweep stopped there.')
    return '\n'.join(lines)


part1_response = build_part1_response(part1_results)
display(Markdown(part1_response))
print(part1_response)


## Part 2 - 2-GPU And 4-GPU `DataParallel` Timing

For Part 2, the prompt asks for timing that includes all training components. The notebook therefore measures full epoch wall time for a 1-GPU baseline, a 2-GPU `DataParallel` run, and a 4-GPU `DataParallel` run, using the successful per-GPU batch sizes from Part 1.

Because the per-GPU batch size stays fixed while the global batch size grows with the number of GPUs, this is a weak-scaling experiment.


In [ ]:
part1_successful_batch_sizes = sorted(part1_successful['per_gpu_batch_size'].astype(int).tolist()) if not part1_successful.empty else []
print('Successful per-GPU batch sizes from Part 1:', part1_successful_batch_sizes)

part2_baseline_results = run_single_gpu_wall_baseline(part1_successful_batch_sizes, device=DEFAULT_DEVICE)
part2_dp_results = run_dataparallel_sweep(part1_successful_batch_sizes, gpu_counts=(2, 4), device=DEFAULT_DEVICE)

part2_baseline_results, part2_dp_results


In [ ]:
def build_table1(part2_baseline: pd.DataFrame, part2_dp: pd.DataFrame) -> pd.DataFrame:
    baseline_ok = part2_baseline[part2_baseline['status'] == 'ok'][['per_gpu_batch_size', 'epoch2_wall_time_sec']].copy()
    baseline_ok = baseline_ok.rename(columns={'epoch2_wall_time_sec': 'single_gpu_wall_time_sec'})
    dp_ok = part2_dp[part2_dp['status'] == 'ok'][['gpu_count', 'per_gpu_batch_size', 'global_batch_size', 'epoch2_wall_time_sec', 'images_per_sec_wall']].copy()
    merged = dp_ok.merge(baseline_ok, on='per_gpu_batch_size', how='left')
    merged['speedup_vs_1gpu'] = merged['single_gpu_wall_time_sec'] / merged['epoch2_wall_time_sec']
    return merged.sort_values(['per_gpu_batch_size', 'gpu_count']).reset_index(drop=True)


def build_part2_response(table1: pd.DataFrame) -> str:
    if table1.empty:
        return 'Run the Part 2 baseline and DataParallel cells first so the notebook can summarize the multi-GPU timings.'
    lines = [
        '## Part 2 Response',
        '',
        'The table below reports epoch-2 wall-clock time and speedup for 2 GPUs and 4 GPUs. The wall-clock timing includes DataLoader wait time, CPU-to-GPU transfer, computation, and synchronization or communication overhead.',
        '',
        format_table_for_markdown(table1, floatfmt='.3f'),
        '',
        'This is weak scaling because the batch size per GPU is fixed while the total batch size increases as more GPUs are used.',
        'If strong scaling were used instead, the global batch size would stay fixed and each GPU would process a smaller local batch. In that setting, speedup would usually be worse because communication and synchronization overhead would take up a larger fraction of each iteration.',
    ]
    return '\n'.join(lines)


table1 = build_table1(part2_baseline_results, part2_dp_results)
display(table1)
part2_response = build_part2_response(table1)
display(Markdown(part2_response))
print(part2_response)


## Part 3 - Compute Time Versus Communication Time

The notebook uses the Part 1 single-GPU compute-only timing as the compute baseline for a given per-GPU batch size. It then subtracts that baseline from the multi-GPU epoch-2 step time to estimate communication time for the 2-GPU and 4-GPU cases.


In [ ]:
def build_table2(part1_results: pd.DataFrame, part2_dp: pd.DataFrame) -> pd.DataFrame:
    single_gpu_compute = part1_results[part1_results['status'] == 'ok'][['per_gpu_batch_size', 'epoch2_step_time_sec']].copy()
    single_gpu_compute = single_gpu_compute.rename(columns={'epoch2_step_time_sec': 'single_gpu_compute_time_sec'})
    dp_ok = part2_dp[part2_dp['status'] == 'ok'][['gpu_count', 'per_gpu_batch_size', 'epoch2_step_time_sec', 'num_steps_epoch2']].copy()
    merged = dp_ok.merge(single_gpu_compute, on='per_gpu_batch_size', how='left')
    merged['compute_time_sec'] = merged['single_gpu_compute_time_sec']
    merged['communication_time_sec'] = (merged['epoch2_step_time_sec'] - merged['single_gpu_compute_time_sec']).clip(lower=0.0)
    return merged[['gpu_count', 'per_gpu_batch_size', 'num_steps_epoch2', 'compute_time_sec', 'communication_time_sec']].sort_values(['per_gpu_batch_size', 'gpu_count']).reset_index(drop=True)


def build_part3_response(table2: pd.DataFrame) -> str:
    if table2.empty:
        return 'Run Parts 1 and 2 first so the notebook can compute the Part 3 breakdown.'
    lines = [
        '## Part 3 Response',
        '',
        'For each per-GPU batch size, I used the single-GPU compute-only epoch-2 time from Part 1 as the compute time. I estimated communication time by subtracting that baseline from the measured multi-GPU epoch-2 step time.',
        '',
        format_table_for_markdown(table2, floatfmt='.3f'),
    ]
    return '\n'.join(lines)


table2 = build_table2(part1_results, part2_dp_results)
display(table2)
part3_response = build_part3_response(table2)
display(Markdown(part3_response))
print(part3_response)


## Part 4 - All-Reduce Model And Bandwidth Utilization

For Part 4, the notebook assumes `DataParallel` follows the ring all-reduce model discussed in class.

Definitions:
- `p`: number of GPUs
- `M`: gradient size in bytes for one model replica
- `B_peak`: assumed peak interconnect bandwidth in bytes/sec
- `T_comm`: measured communication time per epoch from Part 3
- `N_steps`: number of optimizer steps in the measured epoch

Formulas used:
- `T_allreduce_ideal = 2 * (p - 1) / p * M / B_peak`
- `T_allreduce_ideal_epoch = N_steps * 2 * (p - 1) / p * M / B_peak`
- `B_effective = N_steps * 2 * (p - 1) / p * M / T_comm`
- `utilization = B_effective / B_peak`


In [ ]:
def detect_peak_bandwidth_gbps(device_name: str, manual_override_gbps: float | None = MANUAL_PEAK_BW_GBPS) -> float:
    if manual_override_gbps is not None:
        return float(manual_override_gbps)
    normalized = device_name.upper()
    if 'H100' in normalized:
        return 900.0
    if 'A100' in normalized:
        return 600.0
    if 'V100' in normalized:
        return 300.0
    if 'RTX 8000' in normalized:
        return 64.0
    return 64.0


def build_table3(table2: pd.DataFrame, model: nn.Module, reference_device: int = 0) -> pd.DataFrame:
    if table2.empty:
        return pd.DataFrame()
    device_name = torch.cuda.get_device_name(reference_device)
    peak_bandwidth_gbps = detect_peak_bandwidth_gbps(device_name)
    peak_bandwidth_bytes_per_sec = peak_bandwidth_gbps * 1e9
    gradient_bytes = count_trainable_parameters(model) * 4
    rows: list[dict] = []
    for row in table2.itertuples(index=False):
        p = int(row.gpu_count)
        steps = int(row.num_steps_epoch2)
        comm_time = float(row.communication_time_sec)
        total_ring_bytes = steps * (2 * (p - 1) / p) * gradient_bytes
        ideal_epoch_time_sec = total_ring_bytes / peak_bandwidth_bytes_per_sec
        effective_bandwidth_gbps = 0.0 if comm_time <= 0 else total_ring_bytes / comm_time / 1e9
        utilization = 0.0 if peak_bandwidth_gbps <= 0 else effective_bandwidth_gbps / peak_bandwidth_gbps
        rows.append({
            'gpu_count': p,
            'per_gpu_batch_size': int(row.per_gpu_batch_size),
            'num_steps_epoch2': steps,
            'gradient_size_mb': gradient_bytes / 1e6,
            'assumed_peak_bandwidth_gbps': peak_bandwidth_gbps,
            'ideal_allreduce_time_epoch_sec': ideal_epoch_time_sec,
            'effective_bandwidth_gbps': effective_bandwidth_gbps,
            'bandwidth_utilization': utilization,
            'detected_gpu_name': device_name,
        })
    return pd.DataFrame(rows).sort_values(['per_gpu_batch_size', 'gpu_count']).reset_index(drop=True)


def build_part4_response(table3: pd.DataFrame) -> str:
    if table3.empty:
        return 'Run Part 3 first so the notebook can compute the bandwidth-utilization analysis.'
    gpu_name = table3['detected_gpu_name'].iloc[0]
    assumed_peak = table3['assumed_peak_bandwidth_gbps'].iloc[0]
    lines = [
        '## Part 4 Response',
        '',
        'I used the ring all-reduce model discussed in class.',
        '',
        '- Formula for one all-reduce: `T_allreduce_ideal = 2 * (p - 1) / p * M / B_peak`',
        '- Formula for effective bandwidth over one epoch: `B_effective = N_steps * 2 * (p - 1) / p * M / T_comm`',
        '- Formula for bandwidth utilization: `utilization = B_effective / B_peak`',
        '',
        f'The runtime detected GPU model `{gpu_name}` and used an assumed peak bandwidth of {assumed_peak:.1f} GB/s. If your Greene node uses a different interconnect assumption than this default, set `MANUAL_PEAK_BW_GBPS` and rerun this section.',
        '',
        format_table_for_markdown(table3.drop(columns=['detected_gpu_name']), floatfmt='.4f'),
    ]
    return '\n'.join(lines)


bandwidth_reference_model = ResNet18()
table3 = build_table3(table2, bandwidth_reference_model)
display(table3)
part4_response = build_part4_response(table3)
display(Markdown(part4_response))
print(part4_response)
